In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)
import torch


d:\reserach-testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip install -q transformers datasets accelerate sentencepiece



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [6]:
dataset = load_dataset(
    "json",
    data_files="D://reserach-testing//law_ta.jsonl",
    split="train"
)

len(dataset), dataset[0]


Generating train split: 38 examples [00:00, 767.06 examples/s]


(38,
 {'src': 'Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent and moves it for that purpose.',
  'tgt': 'ஒரு நபரின் அனுமதி இல்லாமல் அவரது உடைமையில் உள்ள அசையும் சொத்தை அநியாய நோக்கத்துடன் எடுத்துச் செல்லத் தொடங்கினால் அது திருட்டாகும்.',
  'src_lang': 'eng_Latn',
  'tgt_lang': 'tam_Taml'})

In [8]:
def preprocess_batch(batch):
    # Source language
    tokenizer.src_lang = batch["src_lang"][0]
    # Tokenize source sentences
    model_inputs = tokenizer(
        batch["src"],
        max_length=256,
        truncation=True
    )

    # Target language
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["tgt"],
            max_length=256,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [10]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    save_total_limit=3,
    logging_steps=50,
    fp16=True  # set False if no GPU / not supported
)


In [11]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model("./nllb_finetuned")
tokenizer.save_pretrained("./nllb_finetuned")


NameError: name 'tokenized_dataset' is not defined

In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="D:/reserach-testing/law_ta.jsonl",
    split="train"
)

print(dataset[0])
print(dataset.column_names)


Generating train split: 38 examples [00:00, 7915.35 examples/s]

{'src': 'Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent and moves it for that purpose.', 'tgt': 'ஒரு நபரின் அனுமதி இல்லாமல் அவரது உடைமையில் உள்ள அசையும் சொத்தை அநியாய நோக்கத்துடன் எடுத்துச் செல்லத் தொடங்கினால் அது திருட்டாகும்.', 'src_lang': 'eng_Latn', 'tgt_lang': 'tam_Taml'}
['src', 'tgt', 'src_lang', 'tgt_lang']


In [16]:
def preprocess_batch(batch):
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]
    src_langs = batch["src_lang"]
    tgt_langs = batch["tgt_lang"]

    # ⚠️ Set tokenizer language only once per batch (use the first item)
    tokenizer.src_lang = src_langs[0]

    # Tokenize source sentences
    model_inputs = tokenizer(
        src_texts,
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target sentences
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=256,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [18]:
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Load dataset
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Load model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Choose translation direction
src_lang = "eng_Latn"
tgt_lang = "tam_Taml"
tokenizer.src_lang = src_lang

# Preprocess function
def preprocess_batch(batch):
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]

    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [20]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

# =========================
# Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Example dataset columns: ["src", "tgt"]
print(dataset.column_names)

# =========================
# Load model & tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Set source language
src_lang = "eng_Latn"
tgt_lang = "tam_Taml"
tokenizer.src_lang = src_lang

# =========================
# Preprocessing function
# =========================
def preprocess_batch(batch):
    # Ensure src and tgt are lists
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]

    # Tokenize source
    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target using text_target (for NLLB)
    labels = tokenizer(
        text_target=tgt_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# =========================
# Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])

# =========================
# Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
)

# =========================
# Define Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # or split your dataset
    tokenizer=tokenizer,
)

# =========================
# Start training
# =========================
trainer.train()


['src', 'tgt', 'src_lang', 'tgt_lang']


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [21]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

# =========================
# Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Check dataset columns
print(dataset.column_names)
# Expected: ['src', 'tgt', 'src_lang', 'tgt_lang']

# =========================
# Load model & tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# Preprocessing function (dynamic languages)
# =========================
def preprocess_batch(batch):
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]
    src_langs = batch["src_lang"]
    tgt_langs = batch["tgt_lang"]

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for src, tgt, src_l, tgt_l in zip(src_texts, tgt_texts, src_langs, tgt_langs):
        # Set source language dynamically
        tokenizer.src_lang = src_l

        # Tokenize source
        inputs = tokenizer(
            src,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        # Tokenize target
        labels = tokenizer(
            text_target=tgt,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        input_ids_list.append(inputs["input_ids"])
        attention_mask_list.append(inputs["attention_mask"])
        labels_list.append(labels["input_ids"])

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

# =========================
# Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])

# =========================
# Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
)

# =========================
# Define Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Ideally split train/eval
    tokenizer=tokenizer,
)

# =========================
# Start training
# =========================
trainer.train()

# =========================
# Save the fine-tuned model
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")


['src', 'tgt', 'src_lang', 'tgt_lang']


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [22]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

# =========================
# Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]
print(dataset.column_names)  # should be ['src', 'tgt', 'src_lang', 'tgt_lang']

# =========================
# Load model & tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# Preprocessing function
# =========================
def preprocess_batch(batch):
    # Make sure batch items are lists
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]
    src_langs = batch["src_lang"]
    tgt_langs = batch["tgt_lang"]

    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets dynamically
    labels = []
    for tgt in tgt_texts:
        if tgt is None or tgt.strip() == "":
            # Replace empty target with padding token
            labels.append([tokenizer.pad_token_id])
        else:
            label = tokenizer(
                text_target=tgt,
                max_length=128,
                truncation=True,
                padding="max_length"
            )["input_ids"]
            labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs

# =========================
# Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])

# =========================
# Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
)

# =========================
# Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # ideally split train/eval
    tokenizer=tokenizer,
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Save model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")


['src', 'tgt', 'src_lang', 'tgt_lang']


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [23]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_batch(batch):
    src_texts = batch["src"]
    tgt_texts = batch["tgt"]
    # Replace empty targets with pad token
    tgt_texts = [t if t is not None and t.strip() != "" else tokenizer.pad_token for t in tgt_texts]

    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=tgt_texts,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [24]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

# Load dataset
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Load model & tokenizer
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing function per example (batched=False)
def preprocess_example(example):
    src = example["src"]
    tgt = example["tgt"]
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    # Handle empty target
    if tgt is None or tgt.strip() == "":
        tgt = tokenizer.pad_token

    # Set source language for this example
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target
    labels = tokenizer(
        text_target=tgt,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset (batched=False)
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,  # important
    remove_columns=dataset.column_names
)

print(tokenized_dataset[0])

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # ideally split train/eval
    tokenizer=tokenizer,
)

# Train
trainer.train()

# Save model & tokenizer
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [25]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 1. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]
print("Columns:", dataset.column_names)

# =========================
# 2. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 3. Preprocessing function per example
# =========================
def preprocess_example(example):
    src = example.get("src", None)
    tgt = example.get("tgt", None)
    src_lang = example.get("src_lang", None)
    tgt_lang = example.get("tgt_lang", None)

    # Skip examples with missing data
    if src is None or tgt is None or src.strip() == "" or tgt.strip() == "":
        return None

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target
    labels = tokenizer(
        text_target=tgt,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================
# 4. Tokenize dataset safely
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,       # important: one example at a time
    remove_columns=dataset.column_names,
    # filter out None examples automatically
    keep_in_memory=True
)

print("First tokenized example:", tokenized_dataset[0])

# =========================
# 5. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",  # disable wandb if not used
)

# =========================
# 6. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # ideally split train/eval
    tokenizer=tokenizer,
)

# =========================
# 7. Start training
# =========================
trainer.train()

# =========================
# 8. Save fine-tuned model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")


Columns: ['src', 'tgt', 'src_lang', 'tgt_lang']


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [26]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load dataset
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter invalid examples first
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])

# Load model & tokenizer
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing function
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target safely
    labels = tokenizer(
        text_target=tgt,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Always return a dict
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map dataset
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,           # one example at a time
    remove_columns=dataset.column_names
)

print("First tokenized example:", tokenized_dataset[0])


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


TypeError: 'NoneType' object is not iterable

In [27]:
# =========================
# Full fine-tuning script for NLLB-200-distilled-600M
# Handles English↔Tamil mixed dataset
# =========================

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 1. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter out any empty src/tgt examples
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])

print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 2. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 3. Preprocessing function
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    # Skip empty examples (extra safety)
    if not src or not tgt:
        return None

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize target safely
    labels = tokenizer(
        tgt,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )["input_ids"][0]

    # Prepend forced BOS token ID for target language
    tgt_lang_id = tokenizer.lang_code_to_id[tgt_lang]
    labels = torch.cat([torch.tensor([tgt_lang_id]), labels[1:]])

    model_inputs["labels"] = labels.tolist()

    return model_inputs

# =========================
# 4. Map dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,           # important: process one example at a time
    remove_columns=dataset.column_names
)

print("First tokenized example:", tokenized_dataset[0])

# =========================
# 5. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 6. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",  # disable wandb if not used
)

# =========================
# 7. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# =========================
# 8. Start training
# =========================
trainer.train()

# =========================
# 9. Save fine-tuned model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

# =========================
# 10. Example inference
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml"):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    output_ids = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
        max_length=128
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

example_text = "Theft is committed when someone dishonestly takes property."
translation = translate(example_text)
print("Translation:", translation)


Dataset size after filtering: 38


Map:   0%|          | 0/38 [00:00<?, ? examples/s]


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [30]:
# =========================
# Full working fine-tuning script for NLLB-200-distilled-600M
# Handles English↔Tamil mixed dataset correctly
# =========================

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 1. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter out empty examples
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])
print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 2. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 3. Preprocessing function
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    # Skip empty examples
    if not src or not tgt:
        return None

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Prepend target language prefix (NLLB requires this)
    text_target = f"<<{tgt_lang}>> {tgt}"

    # Tokenize target
    labels = tokenizer(
        text_target,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================
# 4. Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,           # one example at a time (required for mixed languages)
    remove_columns=dataset.column_names
)

print("First tokenized example:", tokenized_dataset[0])

# =========================
# 5. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 6. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# =========================
# 7. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# =========================
# 8. Start training
# =========================
trainer.train()

# =========================
# 9. Save fine-tuned model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

# =========================
# 10. Example inference
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml"):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    forced_prefix = f"<<{tgt_lang}>>"
    output_ids = model.generate(
        **inputs,
        forced_bos_token_id=None,  # NLLB uses prefix in text_target
        max_length=128
    )
    translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Remove language prefix from output
    if translation.startswith(forced_prefix):
        translation = translation[len(forced_prefix):].strip()
    return translation

# Test translation
example_text = "Theft is committed when someone dishonestly takes property."
translation = translate(example_text)
print("Translation:", translation)


Dataset size after filtering: 38
First tokenized example: {'input_ids': [256047, 1617, 1210, 248, 179767, 8852, 9, 3066, 13, 69432, 354, 924, 85250, 6, 429, 2300, 121529, 3559, 452, 43760, 3, 248066, 1123, 148312, 31494, 99797, 540, 14632, 33, 796, 351, 1482, 86952, 248075, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [256047, 57642, 256170, 2054

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [29]:
pip install --upgrade transformers datasets


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
# =========================
# Full working fine-tuning script for NLLB-200-distilled-600M
# Handles English↔Tamil dataset correctly
# =========================

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 1. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter out empty examples
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])
print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 2. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 3. Preprocessing function
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    if not src or not tgt:
        return None

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Prepend target language prefix
    text_target = f"<<{tgt_lang}>> {tgt}"

    # Tokenize target
    labels = tokenizer(
        text_target,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================
# 4. Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,           # one example at a time
    remove_columns=dataset.column_names
)

print("First tokenized example:", tokenized_dataset[0])

# =========================
# 5. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 6. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_steps=50,
    save_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# =========================
# 7. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# =========================
# 8. Start training
# =========================
trainer.train()

# =========================
# 9. Save fine-tuned model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

# =========================
# 10. Example inference
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml"):
    tokenizer.src_lang = src_lang
    # Add target language prefix to guide model
    text_input = text
    inputs = tokenizer(text_input, return_tensors="pt", truncation=True, padding=True)
    output_ids = model.generate(
        **inputs,
        max_length=128
    )
    translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Remove target language prefix if present
    prefix = f"<<{tgt_lang}>>"
    if translation.startswith(prefix):
        translation = translation[len(prefix):].strip()
    return translation

# Test translation
example_text = "Theft is committed when someone dishonestly takes property."
translation = translate(example_text)
print("Translation:", translation)


Dataset size after filtering: 38
First tokenized example: {'input_ids': [256047, 1617, 1210, 248, 179767, 8852, 9, 3066, 13, 69432, 354, 924, 85250, 6, 429, 2300, 121529, 3559, 452, 43760, 3, 248066, 1123, 148312, 31494, 99797, 540, 14632, 33, 796, 351, 1482, 86952, 248075, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [256047, 57642, 256170, 2054

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [32]:
pip install --upgrade transformers datasets


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import transformers
print(transformers.__version__)  # Must be >=4.30


4.57.3


In [35]:
# =========================
# Full working NLLB fine-tuning script
# Transformers 4.57.3 compatible
# =========================

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 1. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter out empty examples
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])
print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 2. Load model & tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 3. Preprocessing function
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    if not src or not tgt:
        return None

    tokenizer.src_lang = src_lang  # set source language

    # Tokenize source
    model_inputs = tokenizer(
        src,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Prepend target language token
    text_target = f"<<{tgt_lang}>> {tgt}"
    labels = tokenizer(
        text_target,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# =========================
# 4. Tokenize dataset
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,          # one example at a time
    remove_columns=dataset.column_names
)

print("First tokenized example:", tokenized_dataset[0])

# =========================
# 5. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 6. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    evaluation_strategy="steps",
    save_steps=200,
    logging_steps=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# =========================
# 7. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

# =========================
# 8. Start training
# =========================
trainer.train()

# =========================
# 9. Save fine-tuned model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

# =========================
# 10. Inference function
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml"):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    output_ids = model.generate(
        **inputs,
        max_length=128
    )
    translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Remove target language prefix if present
    prefix = f"<<{tgt_lang}>>"
    if translation.startswith(prefix):
        translation = translation[len(prefix):].strip()
    return translation

# Example usage
example_text = "Theft is committed when someone dishonestly takes property."
translation = translate(example_text)
print("Translation:", translation)


Dataset size after filtering: 38


Map: 100%|██████████| 38/38 [00:00<00:00, 1465.73 examples/s]

First tokenized example: {'input_ids': [256047, 1617, 1210, 248, 179767, 8852, 9, 3066, 13, 69432, 354, 924, 85250, 6, 429, 2300, 121529, 3559, 452, 43760, 3, 248066, 1123, 148312, 31494, 99797, 540, 14632, 33, 796, 351, 1482, 86952, 248075, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [256047, 57642, 256170, 20545, 7219, 204296, 58101, 198718, 1

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [37]:
pip install --upgrade --force-reinstall transformers



SyntaxError: invalid syntax (174848561.py, line 1)

In [38]:
pip install --upgrade --force-reinstall transformers


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached charset_nor

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Test if it works
args = Seq2SeqTrainingArguments(
    output_dir="./test",
    evaluation_strategy="steps",
    save_steps=10,
    per_device_train_batch_size=1,
    num_train_epochs=1
)
print("Seq2SeqTrainingArguments works!")


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [3]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Test if it works
args = Seq2SeqTrainingArguments(
    output_dir="./test",
    evaluation_strategy="steps",
    save_steps=10,
    per_device_train_batch_size=1,
    num_train_epochs=1
)
print("Seq2SeqTrainingArguments works!")


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [2]:
pip install --upgrade --force-reinstall transformers datasets


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached pyarrow-22.0.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-2.3.3-cp312-cp3

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pyarrow<22,>=7.0, but you have pyarrow 22.0.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
import torch
import transformers
import datasets

print("=== Environment Check ===")
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)


=== Environment Check ===
Python version: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
PyTorch version: 2.9.1+cpu
Transformers version: 4.57.3
Datasets version: 4.4.1


In [5]:
# =========================
# 1. Imports
# =========================
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 2. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])
print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 3. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 4. Preprocess function
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    if not src or not tgt:
        return None

    tokenizer.src_lang = src_lang
    # Tokenize source
    model_inputs = tokenizer(src, max_length=128, truncation=True, padding="max_length")
    
    # Tokenize target with language token
    text_target = f"<<{tgt_lang}>> {tgt}"
    labels = tokenizer(text_target, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================
# 5. Tokenize dataset (CPU-safe)
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,           # process one example at a time
    remove_columns=dataset.column_names
)

# =========================
# 6. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 7. Training arguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    save_total_limit=2,
    per_device_train_batch_size=1,  # CPU -> keep batch small
    per_device_eval_batch_size=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=1,  # CPU -> start with 1
    predict_with_generate=True,
    fp16=False,  # CPU -> cannot use fp16
    logging_steps=10,
    save_steps=20,
    report_to="none",
    evaluation_strategy="steps"
)

# =========================
# 8. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

# =========================
# 9. Start training
# =========================
trainer.train()

# =========================
# 10. Save model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

print("Fine-tuning finished and model saved!")


Dataset size after filtering: 38


Map: 100%|██████████| 38/38 [00:00<00:00, 541.55 examples/s]


TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [6]:
# =========================
# 1. Imports
# =========================
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

# =========================
# 2. Load dataset
# =========================
dataset = load_dataset("json", data_files="D:/reserach-testing/law_ta.jsonl")["train"]

# Filter out any examples with empty src or tgt
dataset = dataset.filter(lambda x: x["src"] and x["tgt"])
print(f"Dataset size after filtering: {len(dataset)}")

# =========================
# 3. Load model and tokenizer
# =========================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# =========================
# 4. Preprocess function (CPU-safe)
# =========================
def preprocess_example(example):
    src = example["src"].strip()
    tgt = example["tgt"].strip()
    src_lang = example["src_lang"]
    tgt_lang = example["tgt_lang"]

    if not src or not tgt:
        return None  # skip bad examples

    # Set source language
    tokenizer.src_lang = src_lang

    # Tokenize source
    model_inputs = tokenizer(src, max_length=128, truncation=True, padding="max_length")

    # Tokenize target with language token
    text_target = f"<<{tgt_lang}>> {tgt}"
    labels = tokenizer(text_target, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================
# 5. Tokenize dataset (CPU-safe)
# =========================
tokenized_dataset = dataset.map(
    preprocess_example,
    batched=False,          # process one example at a time (safe for old datasets)
    remove_columns=dataset.column_names
)

# =========================
# 6. Split train/eval
# =========================
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# =========================
# 7. Training arguments (CPU-friendly)
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    save_total_limit=2,
    per_device_train_batch_size=1,      # CPU -> keep batch small
    per_device_eval_batch_size=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=1,                 # small dataset, start with 1
    predict_with_generate=True,
    fp16=False,                         # CPU -> cannot use fp16
    logging_steps=10,
    save_steps=20,
    report_to="none"                    # disables wandb or other loggers
)

# =========================
# 8. Initialize Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

# =========================
# 9. Start training
# =========================
trainer.train()

# =========================
# 10. Save model & tokenizer
# =========================
trainer.save_model("D:/reserach-testing/nllb_finetuned")
tokenizer.save_pretrained("D:/reserach-testing/nllb_finetuned")

print("Fine-tuning finished and model saved!")


Dataset size after filtering: 38


Map: 100%|██████████| 38/38 [00:00<00:00, 1381.92 examples/s]
C:\Users\sivaj\AppData\Local\Temp\ipykernel_81960\2243458230.py:86: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
d:\reserach-testing\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,10.257800
20,9.497100
30,9.057300


d:\reserach-testing\venv\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Fine-tuning finished and model saved!


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Path where you saved the model
save_path = "D:/reserach-testing/nllb_finetuned"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [10]:
import torch

def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    tokenizer.src_lang = src_lang
    input_ids = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    ).input_ids

    # Get target language ID from model config
    forced_bos_token_id = model.config.lang_code_to_id[tgt_lang]

    outputs = model.generate(
        input_ids,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text


In [11]:
examples = [
    {"src": "Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent.", 
     "src_lang": "eng_Latn", "tgt_lang": "tam_Taml"},
    
    {"src": "ஒரு நபரின் அனுமதி இல்லாமல் அவரது உடைமையில் உள்ள அசையும் சொத்தை அநியாய நோக்கத்துடன் எடுத்துச் செல்லத் தொடங்கினால் அது திருட்டாகும்.", 
     "src_lang": "tam_Taml", "tgt_lang": "eng_Latn"}
]

for ex in examples:
    translation = translate(ex["src"], src_lang=ex["src_lang"], tgt_lang=ex["tgt_lang"])
    print("Source:", ex["src"])
    print("Translation:", translation)
    print("---")



AttributeError: 'M2M100Config' object has no attribute 'lang_code_to_id'

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Path to your saved model
save_path = "D:/reserach-testing/nllb_finetuned"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)

print("Model and tokenizer loaded successfully!")


d:\reserach-testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model and tokenizer loaded successfully!


In [2]:
import torch

def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    tokenizer.src_lang = src_lang
    input_ids = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    ).input_ids

    forced_bos_token_id = model.config.lang_code_to_id[tgt_lang]

    outputs = model.generate(
        input_ids,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [3]:
# Examples to translate
examples = [
    "Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent.",
    "ஒரு நபரின் அனுமதி இல்லாமல் அவரது உடைமையில் உள்ள அசையும் சொத்தை அநியாய நோக்கத்துடன் எடுத்துச் செல்லத் தொடங்கினால் அது திருட்டாகும்."
]

for ex in examples:
    # Detect language by ASCII check
    if all(ord(c) < 128 for c in ex):  # English
        translation = translate(ex, src_lang="eng_Latn", tgt_lang="tam_Taml")
    else:  # Tamil
        translation = translate(ex, src_lang="tam_Taml", tgt_lang="eng_Latn")
    
    print("Source:", ex)
    print("Translation:", translation)
    print("---")


AttributeError: 'M2M100Config' object has no attribute 'lang_code_to_id'

In [4]:
examples = [
    "Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent.",
    "ஒரு நபரின் அனுமதி இல்லாமல் அவரது உடைமையில் உள்ள அசையும் சொத்தை அநியாய நோக்கத்துடன் எடுத்துச் செல்லத் தொடங்கினால் அது திருட்டாகும்."
]

for ex in examples:
    if all(ord(c) < 128 for c in ex):  # English
        translation = translate(ex, src_lang="eng_Latn", tgt_lang="tam_Taml")
    else:  # Tamil
        translation = translate(ex, src_lang="tam_Taml", tgt_lang="eng_Latn")

    print("Source:", ex)
    print("Translation:", translation)
    print("---")


AttributeError: 'M2M100Config' object has no attribute 'lang_code_to_id'

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load your fine-tuned model
save_path = "D:/reserach-testing/nllb_finetuned"
tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)

# Example sentence (English → Tamil)
src_text = "Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent."
tokenizer.src_lang = "eng_Latn"  # source language

# Tokenize
inputs = tokenizer(src_text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)

# Get target language ID
forced_bos_token_id = model.config.lang_code_to_id["tam_Taml"]

# Generate translation
outputs = model.generate(inputs.input_ids, forced_bos_token_id=forced_bos_token_id, max_length=128)

# Decode
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Source:", src_text)
print("Translation:", translation)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AttributeError: 'M2M100Config' object has no attribute 'lang_code_to_id'

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load fine-tuned model
save_path = "D:/reserach-testing/nllb_finetuned"
tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)

# Example sentence
src_text = "Theft is committed when a person dishonestly takes movable property out of someone’s possession without consent."

# Set source and target languages
tokenizer.src_lang = "eng_Latn"   # source language
tgt_lang = "tam_Taml"             # target language

# Tokenize input
inputs = tokenizer(src_text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)

# For M2M100-style models, use tokenizer.get_lang_id() for forced_bos_token_id
forced_bos_token_id = tokenizer.get_lang_id(tgt_lang)

# Generate translation
outputs = model.generate(
    inputs.input_ids,
    forced_bos_token_id=forced_bos_token_id,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

# Decode
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Source:", src_text)
print("Translation:", translation)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AttributeError: NllbTokenizerFast has no attribute get_lang_id

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# =========================
# Load fine-tuned model & tokenizer
# =========================
model_dir = "D:/reserach-testing/nllb_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# =========================
# Translation function
# =========================
def translate(text, src_lang=None, tgt_lang=None, max_length=200):
    if src_lang:
        tokenizer.src_lang = src_lang

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Use get_lang_id from tokenizer
    forced_bos_token_id = tokenizer.get_lang_id(tgt_lang) if tgt_lang else None

    outputs = model.generate(
        inputs.input_ids,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# Test some examples
# =========================
examples = [
    {"src": "What is the punishment for cheating under Sri Lankan Penal Code?", "src_lang": "eng_Latn", "tgt_lang": "tam_Taml"},
    {"src": "ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்ட பிரிவின் கீழ் வருகிறது?", "src_lang": "tam_Taml", "tgt_lang": "eng_Latn"}
]

for ex in examples:
    translation = translate(ex["src"], src_lang=ex["src_lang"], tgt_lang=ex["tgt_lang"])
    print("Source:", ex["src"])
    print("Translation:", translation)
    print("-" * 50)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AttributeError: NllbTokenizerFast has no attribute get_lang_id

In [8]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
import torch

# =========================
# Load fine-tuned model & tokenizer
# =========================
model_dir = "D:/reserach-testing/nllb_finetuned"
tokenizer = NllbTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# =========================
# Translation function
# =========================
def translate(text, src_lang=None, tgt_lang=None, max_length=200):
    if src_lang:
        tokenizer.src_lang = src_lang

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Correct way to get target language ID
    forced_bos_token_id = tokenizer.get_lang_id(tgt_lang) if tgt_lang else None

    outputs = model.generate(
        inputs.input_ids,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# Test examples
# =========================
examples = [
    {"src": "What is the punishment for cheating under Sri Lankan Penal Code?", "src_lang": "eng_Latn", "tgt_lang": "tam_Taml"},
    {"src": "ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்ட பிரிவின் கீழ் வருகிறது?", "src_lang": "tam_Taml", "tgt_lang": "eng_Latn"}
]

for ex in examples:
    translation = translate(ex["src"], src_lang=ex["src_lang"], tgt_lang=ex["tgt_lang"])
    print("Source:", ex["src"])
    print("Translation:", translation)
    print("-" * 50)


AttributeError: NllbTokenizer has no attribute get_lang_id

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# =========================
# Load fine-tuned model & tokenizer
# =========================
model_dir = "D:/reserach-testing/nllb_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# =========================
# Translation function
# =========================
def translate(text, src_lang=None, tgt_lang=None, max_length=200):
    # Prepend source language token
    if src_lang:
        text = f"<<{src_lang}>> {text}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Prepend target language token in generate() using forced_bos_token_id
    # New Transformers versions do not require get_lang_id
    # Instead, add the target language as a prefix in text if needed
    tgt_prefix = f"<<{tgt_lang}>>" if tgt_lang else ""
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove target token if appears
    if decoded.startswith(tgt_prefix):
        decoded = decoded[len(tgt_prefix):].strip()
    return decoded

# =========================
# Test examples
# =========================
examples = [
    {"src": "What is the punishment for cheating under Sri Lankan Penal Code?", "src_lang": "eng_Latn", "tgt_lang": "tam_Taml"},
    {"src": "ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்ட பிரிவின் கீழ் வருகிறது?", "src_lang": "tam_Taml", "tgt_lang": "eng_Latn"}
]

for ex in examples:
    translation = translate(ex["src"], src_lang=ex["src_lang"], tgt_lang=ex["tgt_lang"])
    print("Source:", ex["src"])
    print("Translation:", translation)
    print("-" * 50)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Source: What is the punishment for cheating under Sri Lankan Penal Code?
Translation: << >> What is the punishment for cheating under the Sri Lankan Penal Code?
--------------------------------------------------
Source: ஒரு நபரை ஏமாற்றி பணம் பறிப்பது எந்த சட்ட பிரிவின் கீழ் வருகிறது?
Translation: Which section of the law applies to defrauding a person and taking money from him?
--------------------------------------------------


In [10]:
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=200):
    """
    Translate a single sentence using the fine-tuned NLLB model.
    """
    # Prepend source language token
    text_with_lang = f"<<{src_lang}>> {text}"

    # Tokenize
    inputs = tokenizer(text_with_lang, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Generate translation
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    # Decode
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove target lang token if present at start
    tgt_prefix = f"<<{tgt_lang}>>"
    if decoded.startswith(tgt_prefix):
        decoded = decoded[len(tgt_prefix):].strip()

    return decoded


In [11]:
# Example 1
eng_text1 = "If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft."
translation1 = translate(eng_text1, src_lang="eng_Latn", tgt_lang="tam_Taml")
print("English:", eng_text1)
print("Tamil  :", translation1)
print("-"*50)

# Example 2
eng_text2 = "A person who pawned his watch and takes it back without paying the amount due commits theft even though the watch is his own property."
translation2 = translate(eng_text2)
print("English:", eng_text2)
print("Tamil  :", translation2)
print("-"*50)


English: If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft.
Tamil  : << >> यदि कोई व्यक्ति अपनी घड़ी की मरम्मत के लिए धन का ऋणी है और उसे ज्वैलरी से ले जाता है ताकि उसे कानूनी सुरक्षा से वंचित किया जा सके तो वह चोरी करता है।
--------------------------------------------------
English: A person who pawned his watch and takes it back without paying the amount due commits theft even though the watch is his own property.
Tamil  : << >> कोई व्यक्ति जो अपनी घड़ी का पट्टे पर रखकर उसे वापस ले लेता है और वह भुगतान नहीं करता है, वह चोरी करता है, भले ही घड़ी उसकी संपत्ति हो।
--------------------------------------------------


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_dir = "D:/reserach-testing/nllb_finetuned"  # your fine-tuned model path

# Load fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    # Prepend source language token
    text = f"<<{src_lang}>> {text}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # NLLB uses forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]
    forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
eng_text = "If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft."
translation = translate(eng_text)
print("English:", eng_text)
print("Tamil  :", translation)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [14]:
print(tokenizer.lang_code_to_id.keys())  # should list all language codes like 'eng_Latn', 'tam_Taml'


AttributeError: NllbTokenizerFast has no attribute lang_code_to_id

In [15]:
# =========================
# 1. Imports
# =========================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# =========================
# 2. Load fine-tuned model
# =========================
model_dir = "D:/reserach-testing/nllb_finetuned"  # path to your fine-tuned model

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# Check if tokenizer has lang_code_to_id
if not hasattr(tokenizer, "lang_code_to_id"):
    raise ValueError("Tokenizer does not have 'lang_code_to_id'. Make sure it's an NLLB tokenizer.")

# =========================
# 3. Translation function
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    """
    Translate a single sentence using the fine-tuned NLLB model.
    """
    # Prepend source language token
    text = f"<<{src_lang}>> {text}"

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    # Forced beginning-of-sentence token for target language
    forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]

    # Generate translation
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        early_stopping=True
    )

    # Decode generated tokens
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# 4. Test English → Tamil
# =========================
sentences_eng = [
    "If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft.",
    "A person who pawned his watch and takes it back without paying the amount due commits theft even though the watch is his own property.",
    "Taking an article intending to keep it until money is paid for its return amounts to theft because the taking is dishonest.",
    "Taking a book merely for reading, believing there is implied consent from the owner, does not amount to theft.",
    "Receiving charity from a wife who may reasonably be believed to have authority to give alms does not amount to theft.",
]

for i, eng_text in enumerate(sentences_eng, 1):
    translation = translate(eng_text)
    print(f"{i}. English: {eng_text}")
    print(f"   Tamil  : {translation}")
    print("-"*80)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


ValueError: Tokenizer does not have 'lang_code_to_id'. Make sure it's an NLLB tokenizer.

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_dir = "D:/reserach-testing/nllb_finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    # Prepend source language token
    text = f"<<{src_lang}>> {text}"
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)
    
    # Use convert_tokens_to_ids instead of lang_code_to_id
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(f"<<{tgt_lang}>>")
    
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example test
eng_text = "What is the punishment for cheating under Sri Lankan Penal Code?"
translation = translate(eng_text, src_lang="eng_Latn", tgt_lang="tam_Taml")
print("English:", eng_text)
print("Tamil  :", translation)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


English: What is the punishment for cheating under Sri Lankan Penal Code?
Tamil  : 


In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_dir = "D:/reserach-testing/nllb_finetuned"

# ✅ Fix tokenizer regex issue
tokenizer = AutoTokenizer.from_pretrained(model_dir, fix_mistral_regex=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    # Prepend source language token
    text = f"<<{src_lang}>> {text}"
    
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding="max_length", 
        max_length=max_length
    )
    
    # Convert target language token to id
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(f"<<{tgt_lang}>>")
    
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ✅ Test example
eng_text = "What is the punishment for cheating under Sri Lankan Penal Code?"
translation = translate(eng_text, src_lang="eng_Latn", tgt_lang="tam_Taml")

print("English:", eng_text)
print("Tamil  :", translation)


TypeError: 'tokenizers.pre_tokenizers.Metaspace' object does not support item assignment

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ✅ Use your fine-tuned model path here
finetuned_model = "D:/reserach-testing/nllb_finetuned"

# =========================
# Translation function (same as NLLB example)
# =========================
def translate(model_name, text, src_lang=None, tgt_lang=None, max_length=200):
    tokenizer = AutoTokenizer.from_pretrained(model_name, fix_mistral_regex=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if src_lang:
        text = f"<<{src_lang}>> {text}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Convert target language token to id
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(f"<<{tgt_lang}>>") if tgt_lang else None

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# Test your fine-tuned model
# =========================
eng_text = "What is the punishment for cheating under Sri Lankan Penal Code?"
translation = translate(finetuned_model, eng_text, src_lang="eng_Latn", tgt_lang="tam_Taml")

print("English:", eng_text)
print("Tamil  :", translation)


TypeError: 'tokenizers.pre_tokenizers.Metaspace' object does not support item assignment

In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Fine-tuned model path
finetuned_model = "D:/reserach-testing/nllb_finetuned"

# Translation function
def translate(model_name, text, src_lang=None, tgt_lang=None, max_length=200):
    tokenizer = AutoTokenizer.from_pretrained(model_name)  # ✅ no fix_mistral_regex
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    if src_lang:
        text = f"<<{src_lang}>> {text}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length)

    # Use convert_tokens_to_ids for target language
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(f"<<{tgt_lang}>>") if tgt_lang else None

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# Test fine-tuned model
# =========================
eng_text = "What is the punishment for cheating under Sri Lankan Penal Code?"
translation = translate(finetuned_model, eng_text, src_lang="eng_Latn", tgt_lang="tam_Taml")

print("English:", eng_text)
print("Tamil  :", translation)


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


English: What is the punishment for cheating under Sri Lankan Penal Code?
Tamil  : 


In [20]:
# =========================
# 1. Imports
# =========================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# =========================
# 2. Load your fine-tuned model (once)
# =========================
model_dir = "D:/reserach-testing/nllb_finetuned"  # path to your fine-tuned model

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# =========================
# 3. Translation function
# =========================
def translate(text, src_lang="eng_Latn", tgt_lang="tam_Taml", max_length=128):
    """
    Translate a single sentence from src_lang to tgt_lang using fine-tuned NLLB.
    """
    # Prepend source language token (NLLB style)
    text_input = f"<<{src_lang}>> {text}"

    inputs = tokenizer(
        text_input,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    ).to(device)

    # Force target language token manually
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(f"<<{tgt_lang}>>")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length
    )

    # Decode output tokens
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# 4. Test multiple sentences
# =========================
english_sentences = [
    "If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft.",
    "A person who pawned his watch and takes it back without paying the amount due commits theft even though the watch is his own property.",
    "Taking an article intending to keep it until money is paid for its return amounts to theft because the taking is dishonest.",
    "Taking a book merely for reading, believing there is implied consent from the owner, does not amount to theft.",
    "Receiving charity from a wife who may reasonably be believed to have authority to give alms does not amount to theft.",
    "A man who takes property given by a married woman knowing she lacks authority to give it commits theft if he takes it dishonestly.",
    "If a person takes someone else's property by threat or intimidation, it amounts to robbery.",
    "Cheating someone by dishonest means to gain property is punishable under the law.",
    "Possession of stolen property knowing it to be stolen constitutes an offense.",
    "Forgery of documents to deceive another person is a criminal offense."
]

print("🟢 Testing English → Tamil translations:\n")
for idx, sent in enumerate(english_sentences, 1):
    translation = translate(sent)
    print(f"{idx}. English: {sent}")
    print(f"   Tamil  : {translation}\n")


The tokenizer you are loading from 'D:/reserach-testing/nllb_finetuned' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


🟢 Testing English → Tamil translations:

1. English: If a person owes money for the repair of his watch and takes it from the jeweler to deprive him of lawful security, he commits theft.
   Tamil  : 

2. English: A person who pawned his watch and takes it back without paying the amount due commits theft even though the watch is his own property.
   Tamil  : 

3. English: Taking an article intending to keep it until money is paid for its return amounts to theft because the taking is dishonest.
   Tamil  : 

4. English: Taking a book merely for reading, believing there is implied consent from the owner, does not amount to theft.
   Tamil  : 

5. English: Receiving charity from a wife who may reasonably be believed to have authority to give alms does not amount to theft.
   Tamil  : 

6. English: A man who takes property given by a married woman knowing she lacks authority to give it commits theft if he takes it dishonestly.
   Tamil  : 

7. English: If a person takes someone else's prope

In [1]:
pip install torch torchvision torchaudio transformers accelerate peft bitsandbytes datasets sentencepiece protobuf

  Using cached bitsandbytes-0.48.2-py3-none-win_amd64.whl.metadata (10 kB)
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB 1.3 MB/s eta 0:00:04
    --------------------------------------- 0.1/4.3 MB 1.5 MB/s eta 0:00:03
   -- ------------------------------------- 0.3/4.3 MB 2.3 MB/s eta 0:00:02
   ---- ----------------------------------- 0.5/4.3 MB 3.2 MB/s eta 0:00:02
   -------- ------------------------------- 0.9/4.3 MB 4.6 MB/s eta 0:00:01
   ------------- -------------------------- 1.4/4.3 MB 5.7 MB/s eta 0:00:01
   ----------------- ---------------------- 1.9/4.3 MB 6.4 MB/s eta 0:00:01
   ---------------------- ----------------- 2.4/4.3 MB 6.9 MB/s eta 0:00:01
   -------------------------- ------------- 2.8/4.3 MB 7.3 MB/s eta 0:00:01
   --------------------------- ------------ 3.0/4.3 MB 7.3 MB/s eta 0:00:01
   ---------------------------- ----------- 3.1/4.3 MB 6.3 MB/s eta 0:00:01
   -----------------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import json
import os
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"   # Temporary saves (checkpoints)
final_model_dir = "./nllb-legal-final"    # Final saved model
dataset_file = "nllb_bidir_dataset1.jsonl" # Your uploaded file

# Check GPU
print("=== Hardware Check ===")
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    use_qlora = True
else:
    print("⚠️  No GPU detected. Training will be slow (CPU).")
    use_qlora = False

# ==========================================
# 2. LOAD & PREPARE DATA
# ==========================================
def load_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Dataset file {filename} not found.")
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

print(f"Loading data from {dataset_file}...")
raw_data = load_jsonl(dataset_file)
full_dataset = Dataset.from_list(raw_data)

# Split: 90% Train, 10% Validation
split_data = full_dataset.train_test_split(test_size=0.1, seed=42)
dataset = split_data

# ==========================================
# 3. MODEL & TOKENIZER
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(model_id)

# QLoRA Config (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
) if use_qlora else None

print("Loading base model...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" if use_qlora else None
)

# Enable Gradient Checkpointing (Saves Memory)
model.gradient_checkpointing_enable()

if use_qlora:
    model = prepare_model_for_kbit_training(model)

# LoRA Adapter Config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=32,           # Rank: higher = smarter but more VRAM
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ==========================================
# 4. PREPROCESSING (MIXED LANGUAGES)
# ==========================================
def preprocess_function(examples):
    inputs = []
    targets = []
    for i in range(len(examples["src"])):
        tokenizer.src_lang = examples["src_lang"][i]
        
        # Inputs
        model_inputs = tokenizer(examples["src"][i], max_length=128, truncation=True)
        
        # Targets
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples["tgt"][i], max_length=128, truncation=True)
            
        inputs.append(model_inputs["input_ids"])
        targets.append(labels["input_ids"])
        
    return {"input_ids": inputs, "labels": targets}

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# ==========================================
# 5. SMART RESUME LOGIC
# ==========================================
def get_latest_checkpoint(path):
    if not os.path.exists(path):
        return None
    checkpoints = [d for d in os.listdir(path) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    # Sort by step number
    checkpoints.sort(key=lambda x: int(x.split('-')[1]))
    return os.path.join(path, checkpoints[-1])

last_checkpoint = get_latest_checkpoint(output_dir)
if last_checkpoint:
    print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
else:
    print("🆕 Starting fresh training...")

# ==========================================
# 6. TRAINER SETUP
# ==========================================
args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    
    # Checkpointing (Safety)
    save_strategy="steps", 
    save_steps=50,               # Save every 50 steps
    save_total_limit=2,          # Keep only last 2 to save disk space
    
    # Training Params
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,          # Adjust epochs as needed
    fp16=True if use_qlora else False,
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer,
)

# ==========================================
# 7. EXECUTION
# ==========================================
# This line automatically handles resuming if last_checkpoint is not None
trainer.train(resume_from_checkpoint=last_checkpoint)

print("💾 Saving final model...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Training Complete & Model Saved!")

d:\reserach-testing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Hardware Check ===
⚠️  No GPU detected. Training will be slow (CPU).
Loading data from nllb_bidir_dataset1.jsonl...


JSONDecodeError: Extra data: line 1 column 141 (char 140)

In [3]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
     ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
      --------------------------------------- 0.1/6.2 MB 2.2 MB/s eta 0:00:03
     - -------------------------------------- 0.2/6.2 MB 2.4 MB/s eta 0:00:03
     --- ------------------------------------ 0.5/6.2 MB 3.7 MB/s eta 0:00:02
     ------ --------------------------------- 1.0/6.2 MB 5.1 MB/s eta 0:00:02
     --------- ------------------------------ 1.5/6.2 MB 6.6 MB/s eta 0:00:01
     ------------ --------------------------- 2.0/6.2 MB 7.1 MB/s eta 0:00:01
     --------------- ------------------------ 2.4/6.2 MB 7.8

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch
import json
import os
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
model_id = "facebook/nllb-200-distilled-600M"
output_dir = "./nllb-legal-checkpoints"
final_model_dir = "./nllb-legal-final"
dataset_file = "nllb_bidir_dataset1.jsonl"

print("=== Hardware Check ===")
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    use_qlora = True
else:
    print("⚠️  No GPU detected. Training will be slow (CPU).")
    use_qlora = False

# ==========================================
# 2. ROBUST DATA LOADING (The Fix)
# ==========================================
def load_jsonl(filename):
    data = []
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Dataset file {filename} not found.")
        
    print(f"Reading {filename}...")
    with open(filename, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            # 1. Remove whitespace (newlines, spaces)
            line = line.strip()
            
            # 2. Skip empty lines
            if not line:
                continue
                
            # 3. FIX: Remove trailing comma if present (common error)
            if line.endswith(','):
                line = line[:-1]
                
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"⚠️ Warning: Skipping bad line #{i+1}: {e}")
                # Optional: Print bad line to debug
                # print(f"   Content: {line}")
                continue
                
    print(f"✅ Successfully loaded {len(data)} valid lines.")
    return data

raw_data = load_jsonl(dataset_file)
if len(raw_data) == 0:
    print("❌ Error: No valid data found in file!")
    exit()

full_dataset = Dataset.from_list(raw_data)

# Split: 90% Train, 10% Validation (with check for small data)
if len(full_dataset) > 10:
    split_data = full_dataset.train_test_split(test_size=0.1, seed=42)
    dataset = split_data
else:
    print("⚠️ Dataset too small. Using same data for train/test.")
    dataset = {"train": full_dataset, "test": full_dataset}

# ==========================================
# 3. MODEL & TOKENIZER
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
) if use_qlora else None

print("Loading base model...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" if use_qlora else None
)

model.gradient_checkpointing_enable()
if use_qlora:
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=32, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ==========================================
# 4. PREPROCESSING
# ==========================================
def preprocess_function(examples):
    inputs = []
    targets = []
    for i in range(len(examples["src"])):
        tokenizer.src_lang = examples["src_lang"][i]
        model_inputs = tokenizer(examples["src"][i], max_length=128, truncation=True)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples["tgt"][i], max_length=128, truncation=True)
        inputs.append(model_inputs["input_ids"])
        targets.append(labels["input_ids"])
    return {"input_ids": inputs, "labels": targets}

tokenized_datasets = dataset["train"].map(preprocess_function, batched=True)
if "test" in dataset:
    eval_dataset = dataset["test"].map(preprocess_function, batched=True)
else:
    eval_dataset = None

# ==========================================
# 5. SMART RESUME & TRAINING
# ==========================================
def get_latest_checkpoint(path):
    if not os.path.exists(path):
        return None
    checkpoints = [d for d in os.listdir(path) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda x: int(x.split('-')[1]))
    return os.path.join(path, checkpoints[-1])

last_checkpoint = get_latest_checkpoint(output_dir)
if last_checkpoint:
    print(f"🔄 Resuming from checkpoint: {last_checkpoint}")
else:
    print("🆕 Starting fresh training...")

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="steps", 
    save_steps=50,               
    save_total_limit=2,          
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=5,          
    fp16=True if use_qlora else False,
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    tokenizer=tokenizer,
)

trainer.train(resume_from_checkpoint=last_checkpoint)

print("💾 Saving final model...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print("✅ Training Complete & Model Saved!")

=== Hardware Check ===
⚠️  No GPU detected. Training will be slow (CPU).
Reading nllb_bidir_dataset1.jsonl...
⚠️ Warning: Skipping bad line #4604: Extra data: line 1 column 331 (char 330)
⚠️ Warning: Skipping bad line #4605: Extra data: line 1 column 212 (char 211)
⚠️ Warning: Skipping bad line #4793: Extra data: line 1 column 271 (char 270)
⚠️ Warning: Skipping bad line #4936: Extra data: line 1 column 356 (char 355)
⚠️ Warning: Skipping bad line #8618: Expecting value: line 1 column 1 (char 0)
✅ Successfully loaded 8685 valid lines.
Loading base model...


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434